In [0]:
%sql
use catalog deltalake_catalog;

In [0]:
%sql
DROP TABLE IF EXISTS demo_taxi_200files;
CREATE TABLE demo_taxi_200files (
  vendor_id STRING,
  pickup_datetime TIMESTAMP,
  dropoff_datetime TIMESTAMP,
  passenger_count INT,
  trip_distance DOUBLE,
  pickup_longitude DOUBLE,
  pickup_latitude DOUBLE,
  rate_code_id INT,
  store_and_fwd_flag STRING,
  dropoff_longitude DOUBLE,
  dropoff_latitude DOUBLE,
  payment_type STRING,
  fare_amount DOUBLE,
  extra DOUBLE,
  mta_tax DOUBLE,
  tip_amount DOUBLE,
  tolls_amount DOUBLE,
  total_amount DOUBLE
)
USING DELTA
PARTITIONED BY (vendor_id)
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = false,
  delta.autoOptimize.autoCompact = false
);


In [0]:
# Find the table location from DESCRIBE DETAIL first:
table_location = "dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow"

# List data files (ignore _delta_log)
all_files = [f.path for f in dbutils.fs.ls(table_location) if f.path.endswith(".parquet")]

# Pick first 10 files
files10 = all_files[:10]
print("Using these 10 files:", files10)

# Read them as Parquet
df_10 = spark.read.parquet(*files10)

# Repartition into 200 files and write into your demo Delta table
df_10.repartition(200).write.format("delta").mode("overwrite").saveAsTable("demo_taxi_200files")

Using these 10 files: ['dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b

In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100

count(*)
209


In [0]:
%sql
optimize demo_taxi_200files zorder by (trip_distance)

path,metrics
abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/00892c15-e132-48bf-b581-4ed9d9920a97,"List(46, 600, List(43637595, 96219285, 5.985921323913044E7, 46, 2753523809), List(587250, 8467648, 5322611.6033333335, 600, 3193566962), 4, List(minCubeSize(107374182400), List(0, 0), List(601, 3193571093), 0, List(600, 3193566962), 3, null), null, 0, 1, 601, 1, false, 0, 0, 1758895752971, 1758895776206, 32, 3, null, List(0, 0), null, 18, 18, 244674, 0, null)"


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance)

min(trip_distance),max(trip_distance),file_name
0.0,49.2,part-00025-7eac5e30-c1a9-4ad7-9630-2c23b522b355.c000.snappy.parquet
0.0,0.5,part-00000-91d0b5fe-2041-4cfa-8d26-4e7c4cf663f7.c000.snappy.parquet
0.0,0.5,part-00026-80946cf9-2968-4051-ad35-c7a32f148f7f.c000.snappy.parquet
0.4,0.4,part-00091-9a6608a9-04be-4552-b0b2-602ff50a3673.c000.snappy.parquet
0.5,0.6,part-00001-a793ca2e-c8d2-4eeb-96a9-47608373084e.c000.snappy.parquet
0.5,0.68,part-00027-29638c11-b880-4a33-a015-63637a2a6cb6.c000.snappy.parquet
0.6,0.7,part-00002-f99d1564-cca4-4352-8c61-8fe5a8a310d4.c000.snappy.parquet
0.68,0.83,part-00028-aa22a219-0ebe-4723-b7cb-40ee649be3cd.c000.snappy.parquet
0.7,0.8,part-00003-b998a5a8-3574-40d9-aaec-c7e821c6c37f.c000.snappy.parquet
0.8,1.0,part-00004-bca807a5-519b-480e-b3ca-cf005bfc3874.c000.snappy.parquet


In [0]:
%sql
DROP TABLE IF EXISTS demo_taxi_200files;
CREATE TABLE demo_taxi_200files (
  vendor_id STRING,
  pickup_datetime TIMESTAMP,
  dropoff_datetime TIMESTAMP,
  passenger_count INT,
  trip_distance DOUBLE,
  pickup_longitude DOUBLE,
  pickup_latitude DOUBLE,
  rate_code_id INT,
  store_and_fwd_flag STRING,
  dropoff_longitude DOUBLE,
  dropoff_latitude DOUBLE,
  payment_type STRING,
  fare_amount DOUBLE,
  extra DOUBLE,
  mta_tax DOUBLE,
  tip_amount DOUBLE,
  tolls_amount DOUBLE,
  total_amount DOUBLE
)
USING DELTA
CLUSTER BY (vendor_id, trip_distance)
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = false,
  delta.autoOptimize.autoCompact = false
);


In [0]:
# Find the table location from DESCRIBE DETAIL first:
table_location = "dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow"

# List data files (ignore _delta_log)
all_files = [f.path for f in dbutils.fs.ls(table_location) if f.path.endswith(".parquet")]

# Pick first 10 files
files10 = all_files[:10]
print("Using these 10 files:", files10)

# Read them as Parquet
df_10 = spark.read.parquet(*files10)

# Repartition into 200 files and write into your demo Delta table
df_10.repartition(200).write.format("delta").mode("overwrite").saveAsTable("demo_taxi_200files")

Using these 10 files: ['dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-7dcc09ea-2fc1-491d-a234-3b0ce1db9336-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-812df468-c3fb-405d-84d6-32cb249d8db9-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-a4b4b3de-4231-4c0e-88d6-c14f202ff232-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00000-b3d2db79-4a87-4d35-8a20-a02725fb655f-c001.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-158e21e1-9d7b-44e9-ad15-3ba7e1797de8-c002.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9072b615-4da5-4c0c-b2c0-83f08d1944ac-c000.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-9baf2d71-1e4d-49a0-a131-03c825853637-c003.snappy.parquet', 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow/part-00001-d61d36d1-b864-4dc2-b

In [0]:
%sql
describe detail demo_taxi_200files;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,51ad94c8-900f-4e8f-9d49-6e291daa63af,deltalake_catalog.default.demo_taxi_200files,null,abfss://unitycatalog@ttmystorageaccount001.dfs.core.windows.net/catalog/__unitystorage/catalogs/758280e5-e8ae-48b0-840c-9f10d52cc821/tables/225cf526-2b95-4f75-abb9-de3667fe6ad5,2025-09-26T14:13:41.921Z,2025-09-26T14:15:06.000Z,List(),"List(vendor_id, trip_distance)",33,2708679387,"Map(delta.autoOptimize.autoCompact -> false, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.autoOptimize.optimizeWrite -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-0d57c669-1d12-422e-8dde-cedc0fdff510, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5abd6ee0-60da-42a9-a043-64c0a067337d, collation -> UTF8_BINARY)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
select count(*) from demo_taxi_200files where trip_distance > 100

count(*)
209


In [0]:
%sql
select min(trip_distance), max(trip_distance), _metadata.file_name
from demo_taxi_200files
group by _metadata.file_name
order by min(trip_distance)

min(trip_distance),max(trip_distance),file_name
0.0,0.639,part-00013-61b80a55-751e-43bf-88e2-e3aa4fd8684c.c000.snappy.parquet
0.0,0.52,part-00006-5e684175-6333-4fc2-8ae1-3cb421722d7d.c000.snappy.parquet
0.0,1.9,part-00003-d105acc7-74b2-44a1-9566-e0f14375e653.c000.snappy.parquet
0.6,0.7,part-00010-e822bc16-9d0f-4d24-80bf-25e2fb3552f3.c000.snappy.parquet
0.64,0.8699999999999999,part-00011-e63bcfb5-f6ba-4c4e-8d10-22e9752b680b.c000.snappy.parquet
0.8,0.9,part-00009-5a694d91-bd8c-4907-adf1-0ed2c4d37960.c000.snappy.parquet
0.87,1.089,part-00023-226a0bed-3df8-4f96-975d-b66471ccd990.c000.snappy.parquet
1.0,1.0,part-00015-0b9e3b57-f818-419f-a21a-392a63b4c82b.c000.snappy.parquet
1.09,1.329,part-00016-8c854c8e-cc8e-401e-8d89-5d395c19c162.c000.snappy.parquet
1.1,1.2,part-00012-dd334f7f-48d8-4912-9dc9-76113ac1ed47.c000.snappy.parquet
